<a href="https://colab.research.google.com/github/76213869-sketch/Sem7_MLP2/blob/main/Fase_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FASE 4: MODELOS (MLP SKLEARN + MLP PYTORCH)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# =============================
# 1) CARGA + TARGET
# =============================
df = pd.read_csv('dataset.csv', sep=',', encoding='utf-8-sig')
df['dropout'] = (df['Target'] == 'Dropout').astype(int)

df_model = df.copy()

# =============================
# 2) FEATURE ENGINEERING
# =============================
df_model['aprobacion_rate_1'] = np.where(
    df_model['Curricular units 1st sem (enrolled)'] > 0,
    df_model['Curricular units 1st sem (approved)'] /
    df_model['Curricular units 1st sem (enrolled)'], 0
)

df_model['aprobacion_rate_2'] = np.where(
    df_model['Curricular units 2nd sem (enrolled)'] > 0,
    df_model['Curricular units 2nd sem (approved)'] /
    df_model['Curricular units 2nd sem (enrolled)'], 0
)

df_model['variacion_rendimiento'] = (
    df_model['Curricular units 2nd sem (grade)'] -
    df_model['Curricular units 1st sem (grade)']
)

df_model['carga_total'] = (
    df_model['Curricular units 1st sem (enrolled)'] +
    df_model['Curricular units 2nd sem (enrolled)']
)

df_model['riesgo_financiero'] = (
    (df_model['Tuition fees up to date'] == 0).astype(int) +
    (df_model['Debtor'] == 1).astype(int) +
    (df_model['Scholarship holder'] == 0).astype(int)
)

# =============================
# 3) FEATURES / TARGET
# =============================
features = [
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Tuition fees up to date',
    'Debtor',
    'Scholarship holder',
    'Age at enrollment',
    'Displaced',
    'Gender',
    'GDP',
    'Unemployment rate',
    'Inflation rate',
    'aprobacion_rate_1',
    'aprobacion_rate_2',
    'variacion_rendimiento',
    'carga_total',
    'riesgo_financiero',
]

X = df_model[features].copy()
y = df_model['dropout'].copy()

# =============================
# 4) IMPUTACIÓN
# =============================
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# =============================
# 5) SPLIT
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# =============================
# 6) ESCALADO
# =============================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# =============================
# 7) SMOTE (solo train)
# =============================
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_scaled, y_train
)

print("Datos preparados para entrenamiento")

# ============================================================
# 8) MLP — SCIKIT-LEARN
# ============================================================

from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
    verbose=False
)

mlp.fit(X_train_balanced, y_train_balanced)

print(f"Épocas entrenadas     : {mlp.n_iter_}")
print(f"Clases detectadas     : {mlp.classes_}")
print(f"Mejor validation score: {mlp.best_validation_score_:.4f}")

# Curvas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(mlp.loss_curve_, linewidth=2)
axes[0].set_title("Loss entrenamiento")
axes[0].grid(alpha=0.3)

axes[1].plot(mlp.validation_scores_, linestyle='--', linewidth=2)
axes[1].set_title("Validation score")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fase4_curvas_mlp.png', dpi=150)
plt.show()

# ============================================================
# 9) MLP — PYTORCH
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

X_tr = torch.FloatTensor(X_train_balanced)
y_tr = torch.FloatTensor(y_train_balanced.astype(float))
X_te = torch.FloatTensor(X_test_scaled)
y_te = torch.FloatTensor(y_test.values.astype(float))

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

class MLPDesercion(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

model = MLPDesercion(X_train_balanced.shape[1])

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(Xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_te), y_te)

    train_losses.append(np.mean(batch_losses))
    val_losses.append(val_loss.item())

plt.figure(figsize=(10,5))
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Val", linestyle='--')
plt.legend()
plt.title("Curva PyTorch")
plt.grid(alpha=0.3)
plt.savefig('fase4_pytorch.png', dpi=150)
plt.show()